In [1]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import pandas as pd
import keras 
import cv2
import matplotlib.pyplot as plt
import os
import random
from PIL import Image

In [3]:
base_path = r"C:\Users\chand\anaconda3\envs\Py_code\All\Landmark_Detection"
df = pd.read_csv(os.path.join(base_path, "train.csv"))

In [4]:
df = df.loc[df["id"].str.startswith('00', na=False), :] if "id" in df.columns else df.loc[df["if"].str.startswith('00', na=False), :]
num_classes = len(df["landmark_id"].unique())
num_data = len(df)

In [5]:
data = pd.DataFrame(df["landmark_id"].value_counts())
data.reset_index(inplace=True)
data.columns = ['landmark_id', 'count']
print(data.head())

   landmark_id  count
0        83144     14
1       126637      7
2         9673      6
3        46705      6
4       109169      6


In [6]:
from sklearn.preprocessing import LabelEncoder
from keras.applications.vgg19 import VGG19
from keras.layers import *
from keras import Sequential
import tensorflow as tf
lencoder = LabelEncoder()
lencoder.fit(df["landmark_id"])

Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)Holds the label for each class.","ndarray[int64](3539,)","[ 27, 60, 124,...,135308,135318,135395]"


In [7]:
def label_encode(lbl):
    return lencoder.transform(lbl)

In [8]:
def decode_label(lbl):
    return lencoder.inverse_transform(lbl)

In [9]:
def get_image_from_number(num, dataframe):
    fname = dataframe.iloc[num, 0] 
    label = dataframe.iloc[num, 1] 
    fname = str(fname) + '.jpg'
    
    f1 = fname[0]
    f2 = fname[1]
    f3 = fname[2] 
    
    path = os.path.join(f1, f2, f3, fname)
    im = cv2.imread(os.path.join(base_path, "train", path)) 
    im = cv2.imread(os.path.join(base_path, "train", fname))
    return im, label

In [10]:
print("4 Sample images from random classes")
fig = plt.figure(figsize=(16, 16))
try:
    train_dir = os.path.join(base_path, "train")
    for i in range(1, 5):
        ri = random.choices(os.listdir(train_dir), k=3)
        folder = os.path.join(train_dir, ri[0], ri[1], ri[2])
        random_img = random.choice(os.listdir(folder))
        img = np.array(Image.open(os.path.join(folder, random_img)))
        fig.add_subplot(1, 4, i)
        plt.imshow(img)
        plt.axis('off')
    plt.show()
except Exception as e:
    print(f"Skipping random plot layout check due to pathing configuration: {e}")

4 Sample images from random classes
Skipping random plot layout check due to pathing configuration: [WinError 3] The system cannot find the path specified: 'C:\\Users\\chand\\anaconda3\\envs\\Py_code\\All\\Landmark_Detection\\train'


<Figure size 1600x1600 with 0 Axes>

In [11]:
#parameters
learning_rate = 0.0001
decay_speed = 1e-6
momentum = 0.09
loss_function = "sparse_categorical_crossentropy"
source_model = VGG19(weights=None, include_top=False, input_shape=(224, 224, 3))

In [12]:
model = Sequential()
for layer in source_model.layers:
    model.add(layer)

model.add(Flatten())
model.add(Dropout(0.5))
model.add(Dense(512, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(num_classes, activation="softmax"))

model.summary()
        

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ block1_conv1 (Conv2D)                │ (None, 224, 224, 64)        │           1,792 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block1_conv2 (Conv2D)                │ (None, 224, 224, 64)        │          36,928 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block1_pool (MaxPooling2D)           │ (None, 112, 112, 64)        │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block2_conv1 (Conv2D)                │ (None, 112, 112, 128)       │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block2_conv2 (Conv2D)                │ (None, 112, 112, 128)       │         147,584 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block2_pool (MaxPooling2D)           │ (None, 56, 56, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block3_conv1 (Conv2D)                │ (None, 56, 56, 256)         │         295,168 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block3_conv2 (Conv2D)                │ (None, 56, 56, 256)         │         590,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block3_conv3 (Conv2D)                │ (None, 56, 56, 256)         │         590,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block3_conv4 (Conv2D)                │ (None, 56, 56, 256)         │         590,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block3_pool (MaxPooling2D)           │ (None, 28, 28, 256)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block4_conv1 (Conv2D)                │ (None, 28, 28, 512)         │       1,180,160 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block4_conv2 (Conv2D)                │ (None, 28, 28, 512)         │       2,359,808 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block4_conv3 (Conv2D)                │ (None, 28, 28, 512)         │       2,359,808 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block4_conv4 (Conv2D)                │ (None, 28, 28, 512)         │       2,359,808 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block4_pool (MaxPooling2D)           │ (None, 14, 14, 512)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block5_conv1 (Conv2D)                │ (None, 14, 14, 512)         │       2,359,808 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block5_conv2 (Conv2D)                │ (None, 14, 14, 512)         │       2,359,808 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block5_conv3 (Conv2D)                │ (None, 14, 14, 512)         │       2,359,808 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block5_conv4 (Conv2D)                │ (None, 14, 14, 512)         │       2,359,808 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block5_pool (MaxPooling2D)           │ (None, 7, 7, 512)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼──────────────

 Total params: 34,685,459 (132.31 MB)

 Trainable params: 34,685,459 (132.31 MB)

 Non-trainable params: 0 (0.00 B)

In [13]:
optim1 = keras.optimizers.RMSprop(learning_rate=learning_rate)
model.compile(optimizer=optim1, loss=loss_function, metrics=["accuracy"])

In [14]:
def image_reshape(im, target_size):
    return cv2.resize(im, target_size)

In [15]:
def get_batch(dataframe, start, batch_size):
    image_array = []
    label_array = []

    end_img = start + batch_size
    if end_img > len(dataframe):
        end_img = len(dataframe)

    for idx in range(start, end_img):
        im, label = get_image_from_number(idx, dataframe)
        if im is not None:
            im = image_reshape(im, (224, 224)) / 255.0
            image_array.append(im)
            label_array.append(label)

    if len(label_array) > 0:
        label_array = encode_label(label_array)
    return np.array(image_array), np.array(label_array)

In [16]:
# 4. Training Process
batch_size = 16
epoch_shuffle = True
epochs = 1
# Split dataset
train, val = np.split(df.sample(frac=1, random_state=42), [int(0.8 * len(df))])
print(f"Train size: {len(train)}")
print(f"Validation size: {len(val)}")

Train size: 3232
Validation size: 809


In [17]:
import tensorflow as tf
if not tf.executing_eagerly():
    tf.config.run_functions_eagerly(True)
    

In [18]:
for e in range(epochs):
    print("Epoch : " + str(e + 1) + "/" + str(epochs))
    if epoch_shuffle:
        train = pd.DataFrame(train).sample(frac=1)
    steps = int(np.ceil(len(train) / batch_size))
    for it in range(steps):
        start_idx = it * batch_size
        X_train, y_train = get_batch(train, start_idx, batch_size)
        if X_train is None or len(X_train) == 0:
            continue
        model.train_on_batch(X_train, y_train)

model.save("Model.keras")

Epoch : 1/1


In [20]:
errors = 0
good_preds = []
bad_preds = []

val_steps = int(np.ceil(len(val) / batch_size))

val_df = pd.DataFrame(val)
for it in range(val_steps):
    X_val, y_val = get_batch(val_df, it * batch_size, batch_size) 
    if len(X_val) == 0:
        continue

    result = model.predict(X_val)
    cla = np.argmax(result, axis=1)
    
    for idx, res in enumerate(result):
        global_idx = batch_size * it + idx
        if cla[idx] != y_val[idx]:
            errors += 1
            bad_preds.append((global_idx, cla[idx], res[cla[idx]]))
        else:
            good_preds.append((global_idx, cla[idx], res[cla[idx]]))

In [21]:
plt.figure(figsize=(12, 6))
display_count = min(5, len(good_preds))
for i in range(display_count):
    n = int(good_preds[i][0])
    img, lbl = get_image_from_number(n, val)
    if img is not None:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        plt.subplot(1, display_count, i + 1)
        plt.imshow(img)
        plt.title(f"Pred: {decode_label([good_preds[i][1]])[0]}")
        plt.axis('off')
plt.show()

<Figure size 1200x600 with 0 Axes>

In [22]:
import keras

model = keras.models.load_model(r"C:\Users\chand\anaconda3\envs\Py_code\All\Landmark_Detection\Model.keras")
print("Model loaded successfully!")
model.summary()

C:\Users\chand\anaconda3\envs\Py_code\Lib\site-packages\keras\src\saving\saving_lib.py:843: UserWarning: Skipping variable loading for optimizer 'rm_sprop', because it has 38 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model loaded successfully!


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ block1_conv1 (Conv2D)                │ (None, 224, 224, 64)        │           1,792 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block1_conv2 (Conv2D)                │ (None, 224, 224, 64)        │          36,928 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block1_pool (MaxPooling2D)           │ (None, 112, 112, 64)        │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block2_conv1 (Conv2D)                │ (None, 112, 112, 128)       │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block2_conv2 (Conv2D)                │ (None, 112, 112, 128)       │         147,584 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block2_pool (MaxPooling2D)           │ (None, 56, 56, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block3_conv1 (Conv2D)                │ (None, 56, 56, 256)         │         295,168 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block3_conv2 (Conv2D)                │ (None, 56, 56, 256)         │         590,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block3_conv3 (Conv2D)                │ (None, 56, 56, 256)         │         590,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block3_conv4 (Conv2D)                │ (None, 56, 56, 256)         │         590,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block3_pool (MaxPooling2D)           │ (None, 28, 28, 256)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block4_conv1 (Conv2D)                │ (None, 28, 28, 512)         │       1,180,160 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block4_conv2 (Conv2D)                │ (None, 28, 28, 512)         │       2,359,808 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block4_conv3 (Conv2D)                │ (None, 28, 28, 512)         │       2,359,808 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block4_conv4 (Conv2D)                │ (None, 28, 28, 512)         │       2,359,808 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block4_pool (MaxPooling2D)           │ (None, 14, 14, 512)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block5_conv1 (Conv2D)                │ (None, 14, 14, 512)         │       2,359,808 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block5_conv2 (Conv2D)                │ (None, 14, 14, 512)         │       2,359,808 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block5_conv3 (Conv2D)                │ (None, 14, 14, 512)         │       2,359,808 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block5_conv4 (Conv2D)                │ (None, 14, 14, 512)         │       2,359,808 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block5_pool (MaxPooling2D)           │ (None, 7, 7, 512)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼──────────────

 Total params: 69,370,920 (264.63 MB)

 Trainable params: 34,685,459 (132.31 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 34,685,461 (132.31 MB)